سلول ۱ — Project Path و Settings

In [1]:
import os
import sys
from pathlib import Path
import importlib

# Project root
project_root = Path("..").resolve()

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import config.settings as settings
importlib.reload(settings)

DATA_EMBEDDINGS_PATH = settings.DATA_EMBEDDINGS_PATH
DATA_VECTORSTORE_PATH = settings.DATA_VECTORSTORE_PATH
QDRANT_HOST = settings.QDRANT_HOST
QDRANT_PORT = settings.QDRANT_PORT
COLLECTION_NAME = settings.COLLECTION_NAME

print("Project root:", project_root)
print("Embeddings path:", DATA_EMBEDDINGS_PATH)
print("Vectorstore path:", DATA_VECTORSTORE_PATH)
print("Qdrant host:", QDRANT_HOST)
print("Qdrant port:", QDRANT_PORT)
print("Collection name:", COLLECTION_NAME)

Project root: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance
Embeddings path: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/embeddings
Vectorstore path: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/vectorstore
Qdrant host: localhost
Qdrant port: 6333
Collection name: financial_knowledge_base


In [2]:
!pip install qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [qdrant-client]0m [qdrant-client]

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


سلول ۳ — تست اتصال به Qdrant

In [4]:
from qdrant_client import QdrantClient

VECTORSTORE_DIR = Path(DATA_VECTORSTORE_PATH) / "qdrant_local"
VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)

client = QdrantClient(path=str(VECTORSTORE_DIR))

collections = client.get_collections()

print("Connected to local Qdrant successfully")
print("Vectorstore directory:", VECTORSTORE_DIR)
print(collections)

Connected to local Qdrant successfully
Vectorstore directory: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/vectorstore/qdrant_local
collections=[]


سلول ۴ — خواندن embeddings.json

In [5]:
import json

EMBEDDINGS_FILE = Path(DATA_EMBEDDINGS_PATH) / "komeil_roudi" / "embeddings.json"

with open(EMBEDDINGS_FILE, "r", encoding="utf-8") as f:
    embedded_chunks = json.load(f)

print("Embeddings file:", EMBEDDINGS_FILE)
print("Loaded embedded chunks:", len(embedded_chunks))

first_item = embedded_chunks[0]

print("\nFirst chunk ID:", first_item["chunk_id"])
print("Embedding length:", len(first_item["embedding"]))
print("Title:", first_item["title"])

Embeddings file: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/embeddings/komeil_roudi/embeddings.json
Loaded embedded chunks: 168

First chunk ID: 0A050EE9_1
Embedding length: 3072
Title: سه میزان اصلی در ارزیابی فیلم‌های سینمایی از منظر سواد مالی


سلول ۵ — ساخت Collection در Qdrant

In [6]:
from qdrant_client.models import Distance, VectorParams

VECTOR_SIZE = len(embedded_chunks[0]["embedding"])

# Delete collection if it already exists
existing_collections = [c.name for c in client.get_collections().collections]

if COLLECTION_NAME in existing_collections:
    client.delete_collection(collection_name=COLLECTION_NAME)
    print("Existing collection deleted:", COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=VECTOR_SIZE,
        distance=Distance.COSINE
    )
)

print("Collection created successfully")
print("Collection name:", COLLECTION_NAME)
print("Vector size:", VECTOR_SIZE)
print("Distance metric:", "COSINE")

Collection created successfully
Collection name: financial_knowledge_base
Vector size: 3072
Distance metric: COSINE


سلول ۶ — آماده‌سازی Points برای Qdrant

In [7]:
from qdrant_client.models import PointStruct

points = []

for idx, item in enumerate(embedded_chunks):
    payload = {
        "chunk_id": item["chunk_id"],
        "article_id": item["article_id"],
        "chunk_index": item["chunk_index"],
        "title": item["title"],
        "url": item["url"],
        "text": item["text"],
        "char_count": item["char_count"],
        "embedding_model": item["embedding_model"]
    }

    point = PointStruct(
        id=idx,
        vector=item["embedding"],
        payload=payload
    )

    points.append(point)

print("Prepared points:", len(points))
print("First point ID:", points[0].id)
print("First point vector size:", len(points[0].vector))
print("First point payload keys:", points[0].payload.keys())

Prepared points: 168
First point ID: 0
First point vector size: 3072
First point payload keys: dict_keys(['chunk_id', 'article_id', 'chunk_index', 'title', 'url', 'text', 'char_count', 'embedding_model'])


سلول ۷ — ذخیره Points داخل Qdrant

In [8]:
client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)

print("Points inserted successfully")
print("Inserted points:", len(points))

Points inserted successfully
Inserted points: 168


سلول ۸ — بررسی تعداد رکوردهای Collection

In [10]:
collection_info = client.get_collection(
    collection_name=COLLECTION_NAME
)

count_result = client.count(
    collection_name=COLLECTION_NAME,
    exact=True
)

print("Collection name:", COLLECTION_NAME)
print("Points count:", count_result.count)
print("Status:", collection_info.status)
print("Vector size:", collection_info.config.params.vectors.size)
print("Distance:", collection_info.config.params.vectors.distance)

Collection name: financial_knowledge_base
Points count: 168
Status: green
Vector size: 3072
Distance: Cosine


سلول ۹ — آماده‌سازی اتصال Arvan برای Query Embedding

In [11]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(project_root / ".env")

ARVAN_AI_API_KEY = os.getenv("ARVAN_AI_API_KEY")
ARVAN_EMBEDDING_URL = os.getenv("ARVAN_EMBEDDING_URL")

embedding_client = OpenAI(
    api_key=ARVAN_AI_API_KEY,
    base_url=ARVAN_EMBEDDING_URL
)

print("Arvan embedding client ready")
print("API key loaded:", bool(ARVAN_AI_API_KEY))
print("Gateway URL loaded:", bool(ARVAN_EMBEDDING_URL))

Arvan embedding client ready
API key loaded: True
Gateway URL loaded: True


سلول ۱۰ — ساخت تابع Embedding برای Query

In [12]:
def get_query_embedding(query):
    """
    Create embedding for a user query using Arvan AI Gateway.
    """

    response = embedding_client.embeddings.create(
        model="text-embedding-3-large",
        input=query
    )

    return response.data[0].embedding

سلول ۱۱ — تست جستجو در Vector Store

In [14]:
query = "سواد مالی یعنی چه؟"

query_embedding = get_query_embedding(query)

search_results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_embedding,
    limit=5
)

results = search_results.points

print("Query:", query)
print("Results:", len(results))

for i, result in enumerate(results, start=1):
    print("=" * 80)
    print("Rank:", i)
    print("Score:", result.score)
    print("Chunk ID:", result.payload["chunk_id"])
    print("Title:", result.payload["title"])
    print("URL:", result.payload["url"])
    print()
    print(result.payload["text"][:700])

Query: سواد مالی یعنی چه؟
Results: 5
Rank: 1
Score: 0.5363671958883984
Chunk ID: 5C92BB0D_3
Title: فیل سواد مالی
URL: https://www.fintelligence.ir/Blog/Detail/5C92BB0D/%D9%81%DB%8C%D9%84-%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C

یا سرمایه‌گذاری را تمام آن چیزی معرفی کنیم که او در آینده لازم دارد. روی سخن با کسانی نیست که در برندسازی از نام «سواد مالی» بهره خود را می‌برند؛ مخاطب آن‌هایی هستند که دغدغه تربیت نسل آینده را دارند. در کف هر یک اگر شمعی بدی / اختلاف از گفت‌شان بیرون شدی دانش، نور است و اگر پای تخصص در سواد مالی به میان آید، بسیاری از آسیب‌های برآمده از نگاه بخشی و انحرافی به سواد مالی رخت برمی‌بندد. این تعهدی است که در برابر آیندگان داریم.

داستان فیلِ در تاریکی، از مشهورترین داستان‌های مثنوی مولوی است؛ فیلی که در تاریکی، از ستون و ناودان گرفته تا تخت و بادبزن تلقی و برداشت می‌شود.

از نظرگه گفتشان شد مختلف / آن یکی دالش لقب داد این الف

حکایت سواد مالی و ما هم، حکایت فیل و مردمان هند است.

از آنجا که «سوا
Rank: 2
Score: 0.5262946062717172
Chunk ID: BD65A55B_1
Title: